# Flow Matching Tutorial with Gaussian and Independent Coupling Models

Below is a self-contained Jupyter notebook that walks you through implementing and validating two continuous flow matching (CFM) models on three datasets: a 1D Gaussian mixture, MNIST handwritten digits, and Navier–Stokes vorticity fields. It includes:

- Data setup for the 1D mixture, MNIST, and Navier–Stokes  
- Definition and training of  
  - Gaussian CFM  
  - Independent Coupling CFM  
  - Conditional ICFM for Navier–Stokes next-state prediction using a periodic U-Net  
- Euler solver for generation  
- Validation  
  - Histogram comparison for 1D  
  - Sample visualization for MNIST  
  - Ground-truth and generated vorticity snapshots for Navier–Stokes  


## 1- Introduction

Flow matching frames generative modeling as matching a time-indexed vector field to a known velocity field that pushes samples from a simple prior toward the data distribution.  

We’ll implement two variants:  
- Gaussian CFM, where the conditional is Gaussian with mean $t·x_1$ and evolving variance  
- Independent Coupling CFM, which couples a data sample and a prior via a simple vector field  

Each model is trained by minimizing the mean-squared error between a learned vector field $v_\theta(t, x)$ and a known target velocity $u_t$.  

Dependencies

In [ ]:
import torch
from torch import nn, Tensor
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
import torch.nn.functional as F
from torch.autograd import grad
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import datasets, transforms
import math
from pathlib import Path
import scipy.io as sio

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2- One-Dimensional Mixture of Gaussians dataset

### We define a Mixture of Gaussians as our 1D dataset

In [ ]:
def Mixture_Gaussians(n_samples, n_mg_components, mg_means, mg_var, mg_weights):
   # Step 1: Choose components based on weights
    component_choices = np.random.choice(n_mg_components, size=n_samples, p=mg_weights)
    # Step 2: Sample from selected Gaussians
    x = np.array([
        np.random.normal(mg_means[i], np.sqrt(mg_var[i])) for i in component_choices
    ])

    return x

Let's plot the histogram to get a better feel for the data

In [ ]:
n_mg_components = 2
mg_means = np.array([-6, 6])  # Means of Gaussians
mg_var = np.array([0.4, 0.4])     # Variances
mg_weights = np.array([0.5, 0.5])  # Mixture weights
n_samples = 5000 # Number of samples
x_1D = Mixture_Gaussians(n_samples, n_mg_components, mg_means, mg_var, mg_weights)

plt.hist(x_1D, bins=50)
plt.title("Histogram of Mixture of Gaussians distribution");



Next, we need to define a neural network for the vector field. We use a MLP for the 1D dataset.

In [ ]:
class MLPVectorField(nn.Module):
    def __init__(self, dim: int = 1, h: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, dim))
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        if t.dim() == 0:
            t = t.expand_as(x_t)

        return self.net(torch.cat((t, x_t), -1))
    
    def generation(self, x, n_euler_steps, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).to(x.device)

        for i in range(n_euler_steps):
            x = x + (time_steps[i+1]- time_steps[i]) * self(t=time_steps[i], x_t=x)    

        return x    

### Independent Coupling Flow Matching (ICFM)

$p_t(x|z) = \mathcal{N}(x \mid (1-t)x_0 + t x_1, \sigma^2 I)$

$u_t(x|z) = x_1 - x_0$

Now, we implement the training function for our first flow matching model

In [ ]:
def ICFM_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        x_1 = Tensor(x_1D).unsqueeze(-1)
        x_0 = torch.randn_like(x_1)
        t = torch.rand(len(x_1), 1)
        x_t = (1 - t) * x_0 + t * x_1 + torch.randn_like(x_0) * sigma
        u = x_1 - x_0
        
        optimizer.zero_grad()
        loss = loss_fn(VF(t=t, x_t=x_t), u)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        if epoch % 100 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())

    return VF, loss_hist    

Let's train our first flow matching model

In [ ]:
VF = MLPVectorField(dim=1)
VF_ICFM, loss_hist_ICFM = ICFM_training(VF, n_epochs=2000, sigma=0.1)
plt.plot(loss_hist_ICFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

We randomly generate 5000 samples from the trained model and plot it's histogram to evaluate the model

In [ ]:
x_prior = torch.randn(5000, 1)
n_steps = 32
time_steps = torch.linspace(0, 1.0, n_steps + 1)
x_gen_ICFM = VF_ICFM.generation(x=x_prior, n_euler_steps=n_steps)

plt.hist(x_gen_ICFM.detach().numpy(), bins=50)
plt.title('Histogram of Independent Coupling Flow Matching learned distribution');

### How does the number of Euler steps affect generation?

We now sample from the same trained ICFM model using different numbers of Euler steps. Each run starts from the same Gaussian samples and uses the same histogram bins, so we can compare the effect of the integration step size. Each panel shows the generated histogram together with the true Gaussian mixture density.

To quantify the agreement, we estimate the **KL divergence from the generated distribution to the true distribution** using bin probabilities:

$$D_{\mathrm{KL}}(\hat p\|q)=\sum_{i:\hat p_i>0}\hat p_i\log\frac{\hat p_i}{q_i},$$

where $\hat p_i$ is the fraction of generated samples in bin $i$, and $q_i$ is the probability of that bin under the true mixture. The value above each histogram is reported in **nats**; smaller values indicate closer agreement at the chosen bin resolution.

In [ ]:
def compare_icfm_euler_steps(model, step_counts=(4, 8, 16, 32, 64, 128, 256, 512, 1028),
                             n_samples=5000, n_bins=80, seed=42):
    import math

    parameter = next(model.parameters())
    rng = torch.Generator(device=parameter.device).manual_seed(seed)
    prior = torch.randn(n_samples, 1, device=parameter.device,
                        dtype=parameter.dtype, generator=rng)
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            samples = [model.generation(prior.clone(), n_euler_steps=steps)
                       .detach().cpu().numpy().ravel() for steps in step_counts]
    finally:
        model.train(was_training)
    if not all(np.isfinite(values).all() for values in samples):
        raise ValueError("Non-finite generated samples; check the trained model.")

    means = np.asarray(mg_means, dtype=float)
    stds = np.sqrt(np.asarray(mg_var, dtype=float))
    weights = np.asarray(mg_weights, dtype=float)
    weights = weights / weights.sum()
    low = min(min(values.min() for values in samples), np.min(means - 6 * stds))
    high = max(max(values.max() for values in samples), np.max(means + 6 * stds))
    edges = np.linspace(low, high, n_bins + 1)
    # Include both tail bins so the reference covers the entire real line.
    kl_edges = np.concatenate(([-np.inf], edges, [np.inf]))
    true_mass = np.zeros(len(kl_edges) - 1)
    for mean, std, weight in zip(means, stds, weights):
        for i, (left, right) in enumerate(zip(kl_edges[:-1], kl_edges[1:])):
            a, b = (left - mean) / std, (right - mean) / std
            # Use survival probabilities on the positive side to avoid cancellation.
            if a >= 0:
                mass = 0.5 * (math.erfc(a / math.sqrt(2)) - math.erfc(b / math.sqrt(2)))
            else:
                mass = 0.5 * (math.erfc(-b / math.sqrt(2)) - math.erfc(-a / math.sqrt(2)))
            true_mass[i] += weight * mass
    # Numerical floor only; empty generated bins contribute zero to KL.
    true_mass = np.maximum(true_mass, np.finfo(float).tiny)
    true_mass /= true_mass.sum()

    grid = np.linspace(low, high, 1000)
    true_pdf = sum(weight * np.exp(-0.5 * ((grid - mean) / std)**2)
                   / (std * np.sqrt(2 * np.pi))
                   for mean, std, weight in zip(means, stds, weights))
    ncols = 3
    nrows = math.ceil(len(step_counts) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows),
                             sharex=True, sharey=True, squeeze=False)
    kl_values = {}
    for ax, steps, values in zip(axes.flat, step_counts, samples):
        counts, _ = np.histogram(values, bins=kl_edges)
        generated_mass = counts / counts.sum()
        occupied = generated_mass > 0
        kl = np.sum(generated_mass[occupied]
                    * (np.log(generated_mass[occupied]) - np.log(true_mass[occupied])))
        kl_values[steps] = float(kl)
        ax.hist(values, bins=edges, density=True, alpha=0.65, label="ICFM samples")
        ax.plot(grid, true_pdf, color="black", linewidth=1.5, label="True mixture")
        ax.set_title(f"{steps} Euler steps\nBinned KL(generated || true) = {kl:.4f} nats")
        ax.set_xlabel("x")
        ax.set_ylabel("Density")
        ax.legend()
    for ax in list(axes.flat)[len(step_counts):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()
    return kl_values

icfm_euler_kl = compare_icfm_euler_steps(VF_ICFM)

### Gaussian Flow Matching (GFM)

$p_t(x|x_1) = \mathcal{N}(x \mid t x_1, (t\sigma - t + 1)^2 I)$

$u_t(x|x_1) = \frac{x_1 - (1-\sigma)x}{1 - (1-\sigma)t}$

Here, we implement the training function of our second flow matching model

In [ ]:
def GFM_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        x_1 = Tensor(x_1D).unsqueeze(-1)
        t = torch.rand(len(x_1), 1)
        x_t = t * x_1 +  torch.randn_like(t * x_1) * (t * sigma - t + 1).abs() 
        u = (x_1 - (1-sigma)*x_t) / (1-(1-sigma)*t)
        
        optimizer.zero_grad()
        loss = loss_fn(VF(t=t, x_t=x_t), u)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        if epoch % 100 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())

    return VF, loss_hist    

training the Gaussian Flow Matching model

In [ ]:
VF = MLPVectorField(dim=1)
VF_GFM, loss_hist_GFM = GFM_training(VF, n_epochs=1000, sigma=0.1)
plt.plot(loss_hist_GFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

Evaluating the trained flow matching model

In [ ]:
x_prior = torch.randn(5000, 1)
n_steps = 20
time_steps = torch.linspace(0, 1.0, n_steps + 1)
x_gen_GFM = VF_GFM.generation(x=x_prior, n_euler_steps=n_steps)

plt.hist(x_gen_GFM.detach().numpy(), bins=50)
plt.title('Histogram of Gaussian Flow Matching learned distribution');
 

## 3- MNIST dataset

### As the second dataset, we now use the MNIST handwritten image dataset.

We download the MNIST data and show a few of its images

In [ ]:
# Define transform: ToTensor + Flatten
transform = transforms.Compose([
    transforms.ToTensor(),
])

mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transform)
mnist_train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)



fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img, label = mnist_train[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()


Here, we define a Convolutional Neural Network for the vector field

**Exercise: implement the generation loop.** The U-Net represents the velocity field $v_\theta(t,x)$. Starting from the supplied noise sample $x_0$ at $t_0=t_{\mathrm{start}}$, use the explicit Euler update

$$x_{i+1}=x_i+(t_{i+1}-t_i)\,v_\theta(t_i,x_i),\qquad i=0,\ldots,N-1,$$

where $N$ is `n_euler_steps` and the provided `time_steps` tensor contains $t_0,\ldots,t_N$, ending at $t_{\mathrm{end}}$. In `generation`, write a loop that evaluates the velocity with `self(t=..., x=...)` at the current time and state, then updates `x` at each step. The method should return the final state $x_N$.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8 if out_ch>=8 else 1, out_ch)
        self.norm2 = nn.GroupNorm(8 if out_ch>=8 else 1, out_ch)
        if in_ch != out_ch:
            self.nin = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        else:
            self.nin = nn.Identity()

    def forward(self, x):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.relu(h)
        h = self.conv2(h)
        h = self.norm2(h)
        out = F.relu(h + self.nin(x))
        return out

# -------------------------
# UNet-like architecture (with t as extra channel)
# -------------------------
class UNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64):
        super().__init__()
        # encoder
        self.enc1 = ResBlock(in_ch+1, base_ch)    # +1 channel for t
        self.enc2 = ResBlock(base_ch, base_ch*2)
        self.enc3 = ResBlock(base_ch*2, base_ch*4)
        # decoder
        self.dec3 = ResBlock(base_ch*4 + base_ch*2, base_ch*2)
        self.dec2 = ResBlock(base_ch*2 + base_ch, base_ch)
        self.out_conv = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_ch, in_ch, kernel_size=1)
        )
        self.pool = nn.AvgPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, t, x):
        # x: (B, C, H, W), t: (B, 1, 1, 1) in [0,1]
        B, _, H, W = x.shape
        # expand t to (B,1,H,W)
        t_map = t.expand(B,1,H,W)
        xt = torch.cat([x, t_map], dim=1)   # concat along channel
        # encoder
        e1 = self.enc1(xt)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        # decoder
        d3 = self.upsample(e3)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        d2 = self.upsample(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        out = self.out_conv(d2)
        return out
    

    def generation(self, x, n_euler_steps, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).view(-1,1,1,1).to(x.device)

        # TODO: Implement the Euler integration loop using the equation above.

        return x    


### 3.1 Independent Coupling Flow Matching (ICFM)

$p_t(x|z) = \mathcal{N}(x \mid (1-t)x_0 + t x_1, \sigma^2 I)$

$u_t(x|z) = x_1 - x_0$

In [ ]:
def ICFM_MNIST_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        for (x, _) in mnist_train_loader:
            x_1 = x.to(device)   # (B, 1, 28, 28)
            x_0 = torch.randn_like(x_1).to(device)
            t = torch.rand(len(x_1), 1).to(device)   # (B,)
            t = t.view(-1,1,1,1)    # (B, 1, 1, 1)

             
            # TODO: write down the equations for x_t and u
            x_t = ...
            u = ...     # (B, 1, 28, 28)
            
            optimizer.zero_grad()
            loss = loss_fn(VF(t=t, x=x_t), u)
            loss.backward()
            optimizer.step()
        if epoch % 5 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())
        loss_hist.append(loss.item())    

    return VF, loss_hist    

In [ ]:
VF = UNet().to(device)
VF_ICFM, loss_hist_ICFM = ICFM_MNIST_training(VF, n_epochs=5, sigma=0.1)
plt.plot(loss_hist_ICFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

Finally, we generate random images from the trained Independent Coupling Flow Matching Model

In [ ]:
x_prior = torch.randn(16, 1, 28, 28).to(device)
n_steps = 50
x_gen_ICFM = VF_ICFM.generation(x=x_prior, n_euler_steps=n_steps) 

x_gen_ICFM = x_gen_ICFM.cpu().detach().numpy()
fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img = x_gen_ICFM[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()

### 3.2 Gaussian Flow Matching (GFM)

$p_t(x|x_1) = \mathcal{N}(x \mid t x_1, (t\sigma - t + 1)^2 I)$

$u_t(x|x_1) = \frac{x_1 - (1-\sigma)x}{1 - (1-\sigma)t}$


In this section, you are asked to write the training function of Gaussian Flow Matching for MNIST dataset. In your code, you should train the VF and update the loss history list and return them as outputs (similar to previous training functions). 

In [ ]:
def GFM_MNIST_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        for (x, _) in mnist_train_loader:
            x_1 = x.to(device)   # (B, 1, 28, 28)
            t = torch.rand(len(x_1), 1, 1, 1, device=x_1.device)
            scale = 1 - (1 - sigma) * t
            x_t = t * x_1 + torch.randn_like(x_1) * scale
            u = (x_1 - (1 - sigma) * x_t) / scale

            optimizer.zero_grad()
            loss = loss_fn(VF(t=t, x=x_t), u)
            loss.backward()
            optimizer.step()
        if epoch % 5 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())
        loss_hist.append(loss.item())

    return VF, loss_hist

In [ ]:
VF = UNet().to(device)
VF_GFM, loss_hist_GFM = GFM_MNIST_training(VF, n_epochs=10, sigma=0.1)
plt.plot(loss_hist_GFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

In [ ]:
x_prior = torch.randn(16, 1, 28, 28).to(device)
n_steps = 5
x_gen_GFM = VF_GFM.generation(x=x_prior, n_euler_steps=n_steps) 

x_gen_GFM = x_gen_GFM.cpu().detach().numpy()
fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img = x_gen_GFM[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()

## 4- Navier–Stokes vorticity dataset

We now apply independent-coupling flow matching to the dataset in Practical 1, using its **periodic U-Net as the velocity field**. Like the practical, the task is to predict the next vorticity field given the current field. The model learns a conditional distribution of next states rather than the unconditional distribution of snapshots.

Let $c$ be the normalized current field and $x_1$ the normalized next field. Independently draw $x_0\sim\mathcal N(0,I)$ and a **flow time** $s\sim\mathcal U(0,1)$. We use the straight path

$$x_s=(1-s)x_0+s x_1,\qquad u_s=x_1-x_0,$$

and minimize

$$\mathcal L=\mathbb E\left[\|v_\theta(s,x_s;c)-(x_1-x_0)\|^2\right].$$

This is the zero-added-noise ($\sigma=0$) version of ICFM. Flow time $s$ indexes transport from noise to a next-state sample; it is distinct from the physical simulation time. The conditioning field stays fixed throughout each ODE solve.

In [ ]:
NS_SEED = 42
torch.manual_seed(NS_SEED)
np.random.seed(NS_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(NS_SEED)
ns_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", ns_device)

ns_data_dir = Path("data/navier_stokes")
ns_data_path = ns_data_dir / "ns/NavierStokes_V1e-5_N1200_T20.mat"
if not ns_data_path.is_file():
    from huggingface_hub import hf_hub_download
    ns_data_path = Path(hf_hub_download(
        repo_id="kmario23/standard-pde-benchmark",
        repo_type="dataset",
        filename="ns/NavierStokes_V1e-5_N1200_T20.mat",
        local_dir=ns_data_dir,
    ))

ns_data = sio.loadmat(ns_data_path)
ns_a, ns_u = ns_data["a"], ns_data["u"]
assert ns_a.shape == (1200, 64, 64)
assert ns_u.shape == (1200, 64, 64, 20)
print("Initial conditions:", ns_a.shape, "Future fields:", ns_u.shape)

The dataset implementation below is copied from practical 1 under a new name. Data loaders use `num_workers=0` for compatibility with Windows notebooks.

In [ ]:
class NSFlowPairDataset(Dataset):
    """
    Dataset of one-step vorticity transitions:

        omega_t -> omega_{t+1}

    Each trajectory contains:
        omega_0 = initial_vorticity
        omega_1, ..., omega_20 = vorticity[..., 0:20]

    Therefore each trajectory gives 20 transitions.
    """

    def __init__(
        self,
        initial_vorticity,
        vorticity,
        trajectory_indices,
        mean,
        std,
        spatial_stride=1,
    ):
        self.initial_vorticity = initial_vorticity
        self.vorticity = vorticity
        self.trajectory_indices = np.asarray(trajectory_indices)

        self.mean = mean
        self.std = std
        self.spatial_stride = spatial_stride

        self.n_transitions = vorticity.shape[-1]  # 20


    def __len__(self):
        return len(self.trajectory_indices) * self.n_transitions


    def __getitem__(self, idx):

        # Which trajectory?
        trajectory_position = idx // self.n_transitions

        # Which transition inside that trajectory?
        t = idx % self.n_transitions

        trajectory_idx = self.trajectory_indices[trajectory_position]

        # -------------------------------------------------
        # Construct omega_t
        # -------------------------------------------------

        if t == 0:
            # omega_0 comes from "a"
            x = self.initial_vorticity[
                trajectory_idx,
                ::self.spatial_stride,
                ::self.spatial_stride,
            ]

        else:
            # omega_t is stored at u[..., t-1]
            x = self.vorticity[
                trajectory_idx,
                ::self.spatial_stride,
                ::self.spatial_stride,
                t - 1,
            ]

        # -------------------------------------------------
        # omega_{t+1}
        # -------------------------------------------------

        y = self.vorticity[
            trajectory_idx,
            ::self.spatial_stride,
            ::self.spatial_stride,
            t,
        ]

        # Convert to float32
        x = x.astype(np.float32)
        y = y.astype(np.float32)

        # Normalize
        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std

        # Convert to tensors
        x = torch.from_numpy(x)
        y = torch.from_numpy(y)

        # Add channel dimension:
        #
        # [H, W] -> [1, H, W]
        #
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        return x, y

In [ ]:
NS_STRIDE = 2
NS_BATCH_SIZE = 64
ns_train_indices = np.arange(900)
ns_val_indices = np.arange(900, 1000)
ns_test_indices = np.arange(1000, 1200)

ns_train_a = ns_a[ns_train_indices, ::NS_STRIDE, ::NS_STRIDE].astype(np.float64)
ns_train_u = ns_u[ns_train_indices, ::NS_STRIDE, ::NS_STRIDE, :].astype(np.float64)
ns_count = ns_train_a.size + ns_train_u.size
ns_mean = float((ns_train_a.sum() + ns_train_u.sum()) / ns_count)
ns_var = float((np.square(ns_train_a).sum() + np.square(ns_train_u).sum())
               / ns_count - ns_mean**2)
ns_std = float(np.sqrt(max(ns_var, 0.0)))
assert ns_std > 0, "Training fields must have nonzero variance."
del ns_train_a, ns_train_u

ns_train_dataset, ns_val_dataset, ns_test_dataset = [
    NSFlowPairDataset(ns_a, ns_u, indices, ns_mean, ns_std, NS_STRIDE)
    for indices in (ns_train_indices, ns_val_indices, ns_test_indices)
]
ns_train_loader = DataLoader(ns_train_dataset, batch_size=NS_BATCH_SIZE,
                             shuffle=True, num_workers=0,
                             generator=torch.Generator().manual_seed(NS_SEED))
ns_val_loader = DataLoader(ns_val_dataset, batch_size=NS_BATCH_SIZE,
                           shuffle=False, num_workers=0)
ns_test_loader = DataLoader(ns_test_dataset, batch_size=NS_BATCH_SIZE,
                            shuffle=False, num_workers=0)
print("Pairs (train/validation/test):",
      len(ns_train_dataset), len(ns_val_dataset), len(ns_test_dataset))
print(f"Training mean={ns_mean:.6f}, std={ns_std:.6f}")

The convolution blocks, seven-group normalization, GELU activations, max pooling, bilinear upsampling, skip connections, and channel widths $(7,14,28,56)$ are retained practical 1.

Two changes make the U-Net a conditional velocity field: concatenate $[x_s,c,s]$ as three input channels, and return the single-channel network output directly. The practical's final `input_state + delta` is removed because here we predict the flow velocity, not the next vorticity field. This velocity lives in the space of vorticity arrays; it is not the fluid's two-component physical velocity.

In [ ]:
class NSFlowPeriodicConv2d(nn.Module):
    """
    2D convolution with periodic (circular) boundary conditions.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        bias=True,
    ):
        super().__init__()

        self.padding = kernel_size // 2

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=0,
            bias=bias,
        )

    def forward(self, x):

        # Periodic padding:
        # left, right, top, bottom
        x = F.pad(
            x,
            (
                self.padding,
                self.padding,
                self.padding,
                self.padding,
            ),
            mode="circular",
        )

        return self.conv(x)

class NSFlowDoubleConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(

            NSFlowPeriodicConv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                bias=False,
            ),

            nn.GroupNorm(
                num_groups=7,
                num_channels=out_channels,
            ),

            nn.GELU(),

            NSFlowPeriodicConv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                bias=False,
            ),

            nn.GroupNorm(
                num_groups=7,
                num_channels=out_channels,
            ),

            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)

class NSFlowDown(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.conv = NSFlowDoubleConv(
            in_channels,
            out_channels,
        )

    def forward(self, x):

        x = self.pool(x)
        x = self.conv(x)

        return x

class NSFlowUp(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.conv = NSFlowDoubleConv(
            in_channels + skip_channels,
            out_channels,
        )

    def forward(self, x, skip):

        # Upsample to the same resolution
        # as the corresponding encoder feature.
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        # Skip connection
        x = torch.cat(
            [skip, x],
            dim=1,
        )

        x = self.conv(x)

        return x

class NSFlowUNet(nn.Module):

    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        base_channels=7,
    ):
        super().__init__()

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        # -------------------------
        # Encoder
        # -------------------------

        self.enc1 = NSFlowDoubleConv(
            in_channels,
            c1,
        )

        self.enc2 = NSFlowDown(
            c1,
            c2,
        )

        self.enc3 = NSFlowDown(
            c2,
            c3,
        )

        # -------------------------
        # Bottleneck
        # -------------------------

        self.bottleneck = NSFlowDown(
            c3,
            c4,
        )

        # -------------------------
        # Decoder
        # -------------------------

        self.dec3 = NSFlowUp(
            in_channels=c4,
            skip_channels=c3,
            out_channels=c3,
        )

        self.dec2 = NSFlowUp(
            in_channels=c3,
            skip_channels=c2,
            out_channels=c2,
        )

        self.dec1 = NSFlowUp(
            in_channels=c2,
            skip_channels=c1,
            out_channels=c1,
        )

        # -------------------------
        # Output
        # -------------------------

        self.output = nn.Conv2d(
            c1,
            out_channels,
            kernel_size=1,
        )


    def forward(self, x):

        # x contains the interpolated field, condition, and flow-time map.
        # Encoder
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)

        # Bottleneck
        x4 = self.bottleneck(x3)

        # Decoder
        x = self.dec3(x4, x3)
        x = self.dec2(x,  x2)
        x = self.dec1(x,  x1)

        # Predict flow velocity
        delta = self.output(x)

        return delta

In [ ]:
class NSConditionalVelocity(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet = NSFlowUNet(in_channels=3, out_channels=1, base_channels=7)

    def forward(self, s, x_s, condition):
        # s may be a scalar during sampling or one value per training sample.
        s = torch.as_tensor(s, device=x_s.device, dtype=x_s.dtype)
        s_map = s.reshape(-1, 1, 1, 1).expand(x_s.shape[0], 1, *x_s.shape[-2:])
        return self.unet(torch.cat([x_s, condition, s_map], dim=1))

ns_vf = NSConditionalVelocity().to(ns_device)
print("Trainable parameters:", sum(p.numel() for p in ns_vf.parameters()))

Train with conditional flow matching

Each batch draws fresh Gaussian noise and flow times. We average losses over samples, use AdamW and cosine scheduling as in the practical, and retain the weights with the smallest validation flow-matching loss.

In [ ]:
def ns_flow_epoch(model, loader, optimizer=None, seed=1234):
    training = optimizer is not None
    model.train(training)
    rng = None if training else torch.Generator(device=ns_device).manual_seed(seed)
    total, count = 0.0, 0
    with torch.set_grad_enabled(training):
        for condition, target in loader:
            condition = condition.to(ns_device, dtype=torch.float32)
            x_1 = target.to(ns_device, dtype=torch.float32)
            x_0 = torch.randn(x_1.shape, device=ns_device, generator=rng)
            s = torch.rand((len(x_1), 1, 1, 1), device=ns_device, generator=rng)
            x_s = (1 - s) * x_0 + s * x_1
            velocity_target = x_1 - x_0
            loss = F.mse_loss(model(s, x_s, condition), velocity_target)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite flow-matching loss.")
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total += loss.item() * len(x_1)
            count += len(x_1)
    return total / count

NS_EPOCHS = 5
ns_optimizer = torch.optim.AdamW(ns_vf.parameters(), lr=1e-3, weight_decay=1e-4)
ns_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ns_optimizer, T_max=NS_EPOCHS)
ns_history = {"train": [], "validation": []}
ns_best_loss, ns_best_state = float("inf"), None
for ns_epoch in range(1, NS_EPOCHS + 1):
    ns_train_loss = ns_flow_epoch(ns_vf, ns_train_loader, ns_optimizer)
    ns_val_loss = ns_flow_epoch(ns_vf, ns_val_loader)
    ns_history["train"].append(ns_train_loss)
    ns_history["validation"].append(ns_val_loss)
    if ns_val_loss < ns_best_loss:
        ns_best_loss = ns_val_loss
        ns_best_state = {k: v.detach().cpu().clone() for k, v in ns_vf.state_dict().items()}
    ns_scheduler.step()
    print(f"Epoch {ns_epoch:02d}/{NS_EPOCHS}: "
          f"train={ns_train_loss:.6f}, validation={ns_val_loss:.6f}")

ns_vf.load_state_dict(ns_best_state)
ns_vf.eval()
plt.figure(figsize=(7, 4))
for ns_label, ns_losses in ns_history.items():
    plt.plot(np.arange(1, NS_EPOCHS + 1), ns_losses, label=ns_label)
plt.xlabel("Epoch")
plt.ylabel("Flow-matching MSE")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Generate next states and compare samples with ground truth

Starting from Gaussian noise, integrate $dx_s/ds=v_\theta(s,x_s;c)$ from 0 to 1 with explicit Euler steps. The target field is never supplied to the sampler. De-normalize its output to physical vorticity units using the training statistics.

For several selected transitions of a held-out trajectory, the code below plots the true next field alongside three generated samples conditioned on the same current field. Each sample starts from an independent Gaussian noise draw. Change `trajectory_idx` and `time_steps` below to select the snapshots; a transition index of 0 shows the next state at physical step 1, and 19 shows the next state at step 20.

In [ ]:
@torch.no_grad()
def ns_sample_next(model, condition, n_steps=50, generator=None):
    if n_steps < 1:
        raise ValueError("n_steps must be positive.")
    model.eval()
    model_device = next(model.parameters()).device
    condition = condition.to(model_device, dtype=torch.float32)
    x = torch.randn(condition.shape, device=model_device, generator=generator)
    ds = 1.0 / n_steps
    for step in range(n_steps):
        x = x + ds * model(step * ds, x, condition)
    return x

NS_EULER_STEPS = 50

In [ ]:
# Each row shows one true next field and three independent conditional samples.
trajectory_idx = 1005       # Original trajectory ID: 1000–1199 for test data
time_steps = [0, 5, 10, 19] # Transition indices: 0–19

trajectory_position = np.flatnonzero(ns_test_indices == trajectory_idx).item()
if not time_steps or any(t < 0 or t >= ns_test_dataset.n_transitions for t in time_steps):
    raise ValueError("Select transition indices between 0 and 19.")

ns_plot_rng = torch.Generator(device=ns_device).manual_seed(2026)
ns_fig, ns_axes = plt.subplots(len(time_steps), 4, figsize=(14, 3 * len(time_steps)),
                               squeeze=False, constrained_layout=True)
for ns_row, time_step in enumerate(time_steps):
    dataset_idx = trajectory_position * ns_test_dataset.n_transitions + time_step
    ns_condition, ns_target = ns_test_dataset[dataset_idx]
    ns_conditions = ns_condition.unsqueeze(0).repeat(3, 1, 1, 1)
    ns_predictions = ns_sample_next(ns_vf, ns_conditions,
                                    NS_EULER_STEPS, ns_plot_rng).cpu()
    ns_fields = [(field.squeeze().numpy() * ns_std + ns_mean)
                 for field in (ns_target, *ns_predictions)]
    ns_limit = max(float(np.abs(field).max()) for field in ns_fields)
    ns_titles = [f"Ground truth: step {time_step + 1}", "Sample 1", "Sample 2", "Sample 3"]
    for ns_ax, ns_field, ns_title in zip(ns_axes[ns_row], ns_fields, ns_titles):
        ns_im = ns_ax.imshow(ns_field, origin="lower", cmap="RdBu_r",
                             vmin=-ns_limit, vmax=ns_limit)
        ns_ax.set_title(ns_title)
        ns_ax.set_xticks([])
        ns_ax.set_yticks([])
    ns_fig.colorbar(ns_im, ax=list(ns_axes[ns_row]), shrink=0.8, label="Vorticity")
plt.show()